In [1]:
!pip install kaggle

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mirichoi0218/insurance")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'insurance' dataset.
Path to dataset files: /kaggle/input/insurance


In [3]:
import os
import pandas as pd

In [5]:
os.listdir(path)

['insurance.csv']

In [6]:
df = pd.read_csv(os.path.join(path, 'insurance.csv'))

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [9]:
# Split dataset before encoding
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [10]:
# Encode categorical variables
label_encoders = {}
for col in ['sex', 'smoker', 'region']:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    label_encoders[col] = le  # Store encoders for later use

In [11]:
# Features and target
X_train = train_df.drop(columns=['charges'])
y_train = train_df['charges']
X_test = test_df.drop(columns=['charges'])
y_test = test_df['charges']

In [12]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)

(1070, 6)
(1070,)
(268, 6)


In [13]:
# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [15]:
print(X_train_tensor.shape)

torch.Size([1070, 6])


In [16]:
# Define Neural Network Model
class SimpleNNRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(SimpleNNRegressionModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.network(x)

In [17]:
# Initialize model
input_dim = X_train_tensor.shape[1]
model = SimpleNNRegressionModel(input_dim)

In [18]:
# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [19]:
# Training loop
epochs = 1000
clip_value = 25
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train_tensor)
    loss = criterion(predictions, y_train_tensor)
    loss.backward()

    # torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)

    optimizer.step()

    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [100/1000], Loss: 42524868.0000
Epoch [200/1000], Loss: 32567816.0000
Epoch [300/1000], Loss: 30447918.0000
Epoch [400/1000], Loss: 28137720.0000
Epoch [500/1000], Loss: 26151740.0000
Epoch [600/1000], Loss: 24699910.0000
Epoch [700/1000], Loss: 23867392.0000
Epoch [800/1000], Loss: 23381866.0000
Epoch [900/1000], Loss: 23073504.0000
Epoch [1000/1000], Loss: 22816812.0000


In [20]:
import torch
from torch.utils.data import Dataset, DataLoader

In [21]:
class InsuranceDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
     features = torch.tensor(self.X[idx], dtype=torch.float32)
     target = torch.tensor(self.y.values[idx], dtype=torch.float32)
     return features, target

In [22]:
dataset = InsuranceDataset(X_train, y_train)

In [23]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [24]:
for batch_idx, (features, targets) in enumerate(dataloader):
  print(f"Batch {batch_idx+1} :")
  print("Features : ", features.shape)
  print("Targets : ", targets.shape)
  break

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Batch 1 :
Features :  torch.Size([32, 6])
Targets :  torch.Size([32])


In [25]:
epochs = 1000
for epoch in range(epochs):
    model.train()

    for batch_idx, (batch_X, batch_y) in enumerate(dataloader):
      print(f"Current batch : {batch_idx}")
      optimizer.zero_grad()
      predictions = model(batch_X)
      loss = criterion(predictions, batch_y)
      loss.backward()
      optimizer.step()
      print(f'Batch [{batch_idx+1}/{epochs}], Loss: {loss.item():.4f}')

    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Current batch : 0
Batch [1/1000], Loss: 219256960.0000
Current batch : 1
Batch [2/1000], Loss: 229691456.0000
Current batch : 2
Batch [3/1000], Loss: 219037376.0000
Current batch : 3
Batch [4/1000], Loss: 326236448.0000
Current batch : 4
Batch [5/1000], Loss: 170490064.0000
Current batch : 5
Batch [6/1000], Loss: 225555840.0000
Current batch : 6
Batch [7/1000], Loss: 209198704.0000
Current batch : 7
Batch [8/1000], Loss: 246833024.0000
Current batch : 8
Batch [9/1000], Loss: 120991736.0000
Current batch : 9
Batch [10/1000], Loss: 261835648.0000
Current batch : 10
Batch [11/1000], Loss: 216670176.0000
Current batch : 11
Batch [12/1000], Loss: 92111824.0000
Current batch : 12
Batch [13/1000], Loss: 179920672.0000
Current batch : 13
Batch [14/1000], Loss: 145258864.0000
Current batch : 14
Batch [15/1000], Loss: 296756512.0000
Current batch : 15
Batch [16/1000], Loss: 119192848.0000
Current batch : 16
Batch [17/1000], Loss: 160000480.0000
Current batch : 17
Batch [18/1000], Loss: 151182208

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([14])) that is different to the input size (torch.Size([14, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Streaming output truncated to the last 5000 lines.
Batch [17/1000], Loss: 184401184.0000
Current batch : 17
Batch [18/1000], Loss: 104915416.0000
Current batch : 18
Batch [19/1000], Loss: 156151248.0000
Current batch : 19
Batch [20/1000], Loss: 217016416.0000
Current batch : 20
Batch [21/1000], Loss: 94734568.0000
Current batch : 21
Batch [22/1000], Loss: 139773184.0000
Current batch : 22
Batch [23/1000], Loss: 189801424.0000
Current batch : 23
Batch [24/1000], Loss: 204506800.0000
Current batch : 24
Batch [25/1000], Loss: 163002176.0000
Current batch : 25
Batch [26/1000], Loss: 144577168.0000
Current batch : 26
Batch [27/1000], Loss: 115275184.0000
Current batch : 27
Batch [28/1000], Loss: 114492800.0000
Current batch : 28
Batch [29/1000], Loss: 154582160.0000
Current batch : 29
Batch [30/1000], Loss: 70875224.0000
Current batch : 30
Batch [31/1000], Loss: 171030176.0000
Current batch : 31
Batch [32/1000], Loss: 86542128.0000
Current batch : 32
Batch [33/1000], Loss: 190215264.0000
Cu